# CNN — Exercises

Companion to the note [CNN](CNN.md).

Practise convolution arithmetic (output sizes, parameter counts, receptive fields), the equivariance/invariance story, and implement convolution, im2col, pooling and their backward passes from scratch in NumPy, checked against SciPy and finite differences. No deep-learning framework needed.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×4 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.signal import correlate2d
rng = np.random.default_rng(0)

def num_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar function f at array x (modified in place, restored)."""
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Equivariance vs invariance
*🧠 Concept · ★☆☆*

The note says CNNs exploit **locality** and **translation equivariance**, and that pooling gives **invariance**.

1. Define translation *equivariance* and *invariance* for a map $f$ and a shift operator $T$.
2. Which of these is equivariant, which (approximately) invariant: a conv layer, a max-pool layer, global average pooling, a fully connected layer on the flattened image?
3. Why does the note say convolution is "really cross-correlation", and does it matter for learning?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Equivariant: $f(Tx) = T f(x)$. Invariant: $f(Tx) = f(x)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Equivariant: $f(Tx) = T'f(x)$ (shifting the input shifts the output). Invariant: $f(Tx)=f(x)$.
2. Conv (stride 1, ignoring borders): **equivariant**. Max-pool: equivariant to shifts by multiples of its stride and
   *approximately invariant* to small shifts inside a window. Global average pooling: **invariant** (with circular borders exactly).
   A dense layer on the flattened image: neither, as each pixel has its own weight.
3. True convolution flips the kernel: $\sum_{m,n} I(i-m,j-n)K(m,n)$. Deep-learning libraries skip the flip. Since $K$ is learned,
   the network simply learns the flipped kernel, so nothing changes for learning.

</details>

### Exercise 2 · Why ResNet's skip connection helps
*🧠 Concept · ★★☆*

A residual block computes $y = F(x) + x$. Using the chain rule, write $\frac{\partial y}{\partial x}$ and explain why
a stack of 100 residual blocks is easier to optimise than a plain stack of 100 layers. What happens if $F$ learns to output 0?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compare $\prod_l J_{F_l}$ with $\prod_l (I + J_{F_l})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{\partial y}{\partial x} = J_F + I$. Through $L$ blocks the gradient is $\prod_l (I + J_{F_l})$, which expands to
$I + \sum_l J_{F_l} + \dots$: there is always a direct **identity path** so the gradient cannot vanish just because each
$J_{F_l}$ is small. In a plain network the gradient is $\prod_l J_l$, which shrinks or explodes geometrically.
If $F \equiv 0$ the block is the identity, so adding blocks can never make the *representable* function worse;
the network only has to learn a residual correction (see Exercise 14 for a numerical demonstration).

</details>

### Exercise 3 · Small medical dataset
*🧠 Concept · ★★☆*

You have 600 labelled chest X-rays (2 classes) and a ResNet-50 pretrained on ImageNet. Propose a training strategy.
Address: what to freeze, learning rate, augmentation (which ones are *unsafe* here?), BatchNorm, and how to evaluate.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think transfer learning from [[Training Tricks]] and which augmentations change the label meaning.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Transfer learning**: replace the final layer; first train only the head (backbone frozen), then fine-tune the upper
  stages (or all) with a small learning rate (e.g. 10× lower than the head), possibly with discriminative learning rates.
- **Augmentation**: small rotations, crops, brightness/contrast are fine. Horizontal flips are questionable (heart is on the left:
  "situs inversus" is a real anomaly); strong colour jitter is meaningless on grayscale.
- **BatchNorm**: with tiny batches, keep BN in eval mode (frozen running statistics) or use GroupNorm.
- **Evaluation**: patient-level split (no patient in both train and test), stratified k-fold CV because 600 is small,
  report ROC-AUC/sensitivity rather than only accuracy.

</details>

## Part B · By hand

### Exercise 4 · Output sizes
*✍️ By hand · ★☆☆*

Use $\left\lfloor\frac{W-F+2P}{S}\right\rfloor+1$ to compute the spatial output size:

(a) $W=32, F=5, P=2, S=1$ · (b) AlexNet's first layer: $W=227, F=11, P=0, S=4$ · (c) $W=28, F=3, P=1, S=2$ · (d) a $2\times2$ max-pool with stride 2 on $W=55$.

In [ ]:
out_a = None
out_b = None
out_c = None
out_d = None

_o = lambda W, F, P, S: len(range(0, W + 2 * P - F + 1, S))   # count valid window positions
check('(a)', out_a, _o(32, 5, 2, 1)); check('(b)', out_b, _o(227, 11, 0, 4))
check('(c)', out_c, _o(28, 3, 1, 2)); check('(d)', out_d, _o(55, 2, 0, 2))

<details>
<summary><b>💡 Hint</b></summary>

(c): $(28-3+2)/2 = 13.5$, floor it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $(32-5+4)/1+1 = 32$ ("same" padding). (b) $(227-11)/4+1 = 55$. (c) $\lfloor 27/2\rfloor+1 = 14$. (d) $\lfloor 53/2\rfloor+1 = 27$.

```python
out_a, out_b, out_c, out_d = 32, 55, 14, 27
```

</details>

### Exercise 5 · Parameter count: conv vs dense
*✍️ By hand · ★☆☆*

Input: a $32\times32\times3$ image. Output: a $32\times32\times64$ feature map.

(a) How many parameters (with biases) does a $3\times3$ conv layer need? (b) How many would a dense layer mapping the
flattened input to the flattened output need (with biases)? (c) The ratio dense/conv.

In [ ]:
p_conv = None
p_dense = None
ratio = None

_pc = 64 * 3 * 3 * 3 + 64
_pd = (32 * 32 * 3) * (32 * 32 * 64) + 32 * 32 * 64
check('conv params', p_conv, _pc); check('dense params', p_dense, _pd); check('ratio', ratio, _pd / _pc, tol=1)

<details>
<summary><b>💡 Hint</b></summary>

Conv: $C_{in} C_{out} k^2 + C_{out}$, independent of the image size. Dense: $(\text{in})(\text{out}) + \text{out}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $3\cdot64\cdot9 + 64 = 1792$. (b) $3072\cdot65536 + 65536 = 201{,}392{,}128$. (c) $\approx 112{,}384\times$ more.
Weight sharing and locality are what make CNNs practical.

```python
p_conv = 3 * 64 * 9 + 64
p_dense = 3072 * 65536 + 65536
ratio = p_dense / p_conv
```

</details>

### Exercise 6 · Convolution by hand
*✍️ By hand · ★☆☆*

Compute the "valid" cross-correlation (stride 1, no padding) of

$I = \begin{pmatrix}1&2&0&1\\0&1&3&2\\1&0&1&0\\2&1&0&1\end{pmatrix}$ with $K = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$.

Store the $3\times3$ result in `conv_hand`.

In [ ]:
I_hand = np.array([[1, 2, 0, 1], [0, 1, 3, 2], [1, 0, 1, 0], [2, 1, 0, 1]], float)
K_hand = np.array([[1, 0], [0, -1]], float)
conv_hand = None  # 3x3 list or array

check('valid cross-correlation', conv_hand, correlate2d(I_hand, K_hand, mode='valid'))

<details>
<summary><b>💡 Hint</b></summary>

Each output is $I(i,j) - I(i+1,j+1)$: top-left minus bottom-right of every $2\times2$ window.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$O(i,j) = I(i,j) - I(i+1,j+1)$:
row 0: $1-1=0,\ 2-3=-1,\ 0-2=-2$; row 1: $0-0=0,\ 1-1=0,\ 3-0=3$; row 2: $1-1=0,\ 0-0=0,\ 1-1=0$.
This kernel is a diagonal edge detector.

```python
I_hand = np.array([[1, 2, 0, 1], [0, 1, 3, 2], [1, 0, 1, 0], [2, 1, 0, 1]], float)
K_hand = np.array([[1, 0], [0, -1]], float)
conv_hand = [[0, -1, -2], [0, 0, 3], [0, 0, 0]]
```

</details>

### Exercise 7 · Receptive field
*✍️ By hand · ★★☆*

The receptive field grows with depth. With jump $j_l = j_{l-1} s_l$ (start $j_0 = 1$) and
$r_l = r_{l-1} + (k_l - 1) j_{l-1}$ (start $r_0 = 1$), compute the receptive field of one output unit for

(a) three $3\times3$ convs, stride 1; (b) conv $3\times3$/s1 → max-pool $2\times2$/s2 → conv $3\times3$/s1;
(c) why do VGG-style networks stack two $3\times3$ convs instead of one $5\times5$? Count weights for $C_{in}=C_{out}=64$ (no bias) and store the two numbers.

In [ ]:
rf_a = None
rf_b = None
w_two_3x3 = None
w_one_5x5 = None

def _rf(layers):   # brute force: which input pixels influence output pixel 0 (1-D, enough by symmetry)
    idx = {0}
    for k, s in reversed(layers):
        idx = {i * s + m for i in idx for m in range(k)}
    return max(idx) - min(idx) + 1
check('(a) RF', rf_a, _rf([(3, 1)] * 3)); check('(b) RF', rf_b, _rf([(3, 1), (2, 2), (3, 1)]))
check('two 3x3', w_two_3x3, 2 * 64 * 64 * 9); check('one 5x5', w_one_5x5, 64 * 64 * 25)

<details>
<summary><b>💡 Hint</b></summary>

(a) Each stride-1 $3\times3$ adds 2. (b) After the pool the jump is 2, so the last conv adds $(3-1)\cdot 2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $r = 1+2+2+2 = 7$. (b) conv: $r=3, j=1$; pool: $r = 3 + 1\cdot1 = 4, j = 2$; conv: $r = 4 + 2\cdot2 = 8$.
(c) Two $3\times3$ layers see $5\times5$ (like one $5\times5$) but cost $2\cdot 9 C^2 = 73{,}728$ vs $25C^2 = 102{,}400$
weights, **and** add an extra non-linearity.

```python
rf_a, rf_b = 7, 8
w_two_3x3, w_one_5x5 = 2 * 9 * 64 * 64, 25 * 64 * 64
```

</details>

## Part C · Code from scratch

### Exercise 8 · Naive 2-D cross-correlation
*💻 Code · ★☆☆*

Implement `conv2d(I, K, stride=1, pad=0)` for a single-channel image with explicit loops (zero padding).
It must agree with `scipy.signal.correlate2d(..., mode='valid')` for `pad=0, stride=1`, and with the output-size formula otherwise.

In [ ]:
def conv2d(I, K, stride=1, pad=0):
    # TODO: zero-pad I, then slide K with the given stride
    return None

img = rng.normal(size=(9, 9)); ker = rng.normal(size=(3, 3))
out_valid = conv2d(img, ker)
out_strided = conv2d(img, ker, stride=2, pad=1)

check('valid conv = scipy', out_valid, correlate2d(img, ker, mode='valid'))
check('stride 2, pad 1 = subsampled scipy "same"', out_strided, correlate2d(img, ker, mode='same')[::2, ::2])

<details>
<summary><b>💡 Hint 1</b></summary>

`Ip = np.pad(I, pad)`; output size `(H + 2*pad - kh)//stride + 1`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`out[i, j] = np.sum(Ip[i*s:i*s+kh, j*s:j*s+kw] * K)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Strided convolution equals the stride-1 output subsampled, which is what the second check uses.

```python
def conv2d(I, K, stride=1, pad=0):
    Ip = np.pad(I, pad)
    kh, kw = K.shape
    Ho = (Ip.shape[0] - kh) // stride + 1
    Wo = (Ip.shape[1] - kw) // stride + 1
    out = np.zeros((Ho, Wo))
    for i in range(Ho):
        for j in range(Wo):
            out[i, j] = np.sum(Ip[i*stride:i*stride+kh, j*stride:j*stride+kw] * K)
    return out

img = rng.normal(size=(9, 9)); ker = rng.normal(size=(3, 3))
out_valid = conv2d(img, ker)
out_strided = conv2d(img, ker, stride=2, pad=1)
```

</details>

### Exercise 9 · Sobel edge detector
*💻 Code · ★☆☆*

Build a $16\times16$ image that is 0 on the left half and 1 on the right half. Apply the vertical-edge Sobel kernel
$K = \begin{pmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{pmatrix}$ with your `conv2d` (valid). In which output columns is the
response non-zero? Store them as a sorted list in `edge_cols`.

In [ ]:
sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
step_img = None   # TODO
edge_cols = None

_s = np.zeros((16, 16)); _s[:, 8:] = 1
check('edge columns', edge_cols, sorted(set(np.nonzero(correlate2d(_s, np.array([[-1,0,1],[-2,0,2],[-1,0,1.]]), 'valid'))[1])))

<details>
<summary><b>💡 Hint</b></summary>

Output column $j$ covers input columns $j, j+1, j+2$. The response is non-zero only where those straddle the step at column 8.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Output column $j$ sees input columns $j..j+2$; the window contains both a 0 and a 1 column iff $j \in \{6, 7\}$,
each with response $4$ (= $1+2+1$). Everywhere else the image is locally constant and the zero-sum kernel gives 0.

```python
sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
step_img = np.zeros((16, 16)); step_img[:, 8:] = 1
resp = conv2d(step_img, sobel)
edge_cols = sorted(set(np.nonzero(np.abs(resp) > 1e-12)[1].tolist()))
print(edge_cols, resp[0, 5:9])
```

</details>

### Exercise 10 · Multi-channel convolution via im2col
*💻 Code · ★★☆*

Implement a vectorised conv layer `conv_forward(X, W, b)` with `X` of shape `(N, C, H, W)`, kernels `W` of shape
`(F, C, k, k)`, bias `b` of shape `(F,)`, stride 1, no padding, using **im2col**: unfold every $C\times k\times k$ patch into
a row, then do one matrix multiplication.

In [ ]:
def im2col(X, k):
    # TODO: return (N, Ho, Wo, C*k*k)
    return None

def conv_forward(X, W, b):
    # TODO: return (N, F, Ho, Wo)
    return None

Xc = rng.normal(size=(2, 3, 7, 7)); Wc = rng.normal(size=(4, 3, 3, 3)); bc = rng.normal(size=4)
Yc = conv_forward(Xc, Wc, bc)

_ref = np.array([[sum(correlate2d(Xc[n, c], Wc[f, c], 'valid') for c in range(3)) + bc[f] for f in range(4)] for n in range(2)])
check('im2col conv = sum of scipy correlations', Yc, _ref)

<details>
<summary><b>💡 Hint 1</b></summary>

Loop over the $k^2$ kernel offsets (not over output pixels): `X[:, :, di:di+Ho, dj:dj+Wo]` gives all patches' $(di,dj)$ entry at once.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Stack these into shape `(N, C, k, k, Ho, Wo)`, transpose to `(N, Ho, Wo, C, k, k)` and reshape. Then `cols @ W.reshape(F, -1).T + b`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

im2col turns convolution into a GEMM, which is how cuDNN/BLAS make it fast. The memory cost is a $k^2$-fold
duplication of the input.

```python
def im2col(X, k):
    N, C, H, Wd = X.shape
    Ho, Wo = H - k + 1, Wd - k + 1
    cols = np.empty((N, C, k, k, Ho, Wo))
    for di in range(k):
        for dj in range(k):
            cols[:, :, di, dj] = X[:, :, di:di+Ho, dj:dj+Wo]
    return cols.transpose(0, 4, 5, 1, 2, 3).reshape(N, Ho, Wo, C * k * k)

def conv_forward(X, W, b):
    F, C, k, _ = W.shape
    cols = im2col(X, k)
    out = cols @ W.reshape(F, -1).T + b         # (N, Ho, Wo, F)
    return out.transpose(0, 3, 1, 2)

Xc = rng.normal(size=(2, 3, 7, 7)); Wc = rng.normal(size=(4, 3, 3, 3)); bc = rng.normal(size=4)
Yc = conv_forward(Xc, Wc, bc)
```

</details>

### Exercise 11 · Conv backward: gradient w.r.t. the kernel
*💻 Code · ★★★*

For a single-channel valid convolution $O = I \star K$ and upstream gradient $G = \partial L/\partial O$, show that
$\frac{\partial L}{\partial K} = I \star G$ (valid cross-correlation of the input with $G$) and
$\frac{\partial L}{\partial I} = G_{\text{pad}} * K$ (a *full* convolution, i.e. correlation with the flipped kernel).
Implement `conv_backward(I, K, G)` returning `(dI, dK)` and verify with finite differences of $L = \sum O \odot G$.

In [ ]:
def conv_backward(I, K, G):
    # TODO: return dI, dK
    return None, None

I_b = rng.normal(size=(6, 6)); K_b = rng.normal(size=(3, 3)); G_b = rng.normal(size=(4, 4))
dI_b, dK_b = conv_backward(I_b, K_b, G_b)

_L = lambda: np.sum(correlate2d(I_b, K_b, 'valid') * G_b)
check('dK (finite differences)', dK_b, num_grad(_L, K_b), tol=1e-5)
check('dI (finite differences)', dI_b, num_grad(_L, I_b), tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

$O_{ij} = \sum_{mn} I_{i+m,j+n}K_{mn}$, so $\partial L/\partial K_{mn} = \sum_{ij} G_{ij} I_{i+m,j+n}$: a valid correlation of $I$ with $G$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\partial L/\partial I_{pq} = \sum_{ij} G_{ij} K_{p-i,q-j}$: `correlate2d(G, K[::-1, ::-1], 'full')` (or `scipy.signal.convolve2d(G, K, 'full')`).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{\partial L}{\partial K_{mn}} = \sum_{i,j} G_{ij}\,I_{i+m,\,j+n}$ (correlate $I$ with $G$).
$\frac{\partial L}{\partial I_{pq}} = \sum_{i,j} G_{ij}\,K_{p-i,\,q-j}$, which is a full *convolution* of $G$ with $K$:
the backward pass of a conv is again a conv, with the kernel flipped. This is why "transposed convolution" is used for upsampling.

```python
def conv_backward(I, K, G):
    dK = correlate2d(I, G, 'valid')
    dI = correlate2d(G, K[::-1, ::-1], 'full')
    return dI, dK

I_b = rng.normal(size=(6, 6)); K_b = rng.normal(size=(3, 3)); G_b = rng.normal(size=(4, 4))
dI_b, dK_b = conv_backward(I_b, K_b, G_b)
```

</details>

### Exercise 12 · Max-pooling forward and backward
*💻 Code · ★★☆*

Implement `maxpool_forward(X, p)` ($p\times p$ windows, stride $p$, `X` of shape `(H, W)` divisible by $p$) and
`maxpool_backward(X, p, G)` that routes each upstream gradient to the argmax of its window (zeros elsewhere).

In [ ]:
def maxpool_forward(X, p=2):
    return None

def maxpool_backward(X, p, G):
    return None

X_mp = rng.normal(size=(6, 6)); G_mp = rng.normal(size=(3, 3))
Y_mp = maxpool_forward(X_mp, 2)
dX_mp = maxpool_backward(X_mp, 2, G_mp)

check('forward = max over blocks', Y_mp, X_mp.reshape(3, 2, 3, 2).max(axis=(1, 3)))
_Lmp = lambda: np.sum(X_mp.reshape(3, 2, 3, 2).max(axis=(1, 3)) * G_mp)
check('backward (finite differences)', dX_mp, num_grad(_Lmp, X_mp), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

Reshape to `(H/p, p, W/p, p)`; the max is over axes 1 and 3. For backward, build a mask `X == upsampled max` and multiply by the upsampled `G` (`np.kron` or `np.repeat`).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Max-pooling has no parameters; its gradient is a *switch*: only the winning input of each window receives gradient.
(With ties the mask approach sends gradient to all tied entries; frameworks pick one.)

```python
def maxpool_forward(X, p=2):
    H, W = X.shape
    return X.reshape(H // p, p, W // p, p).max(axis=(1, 3))

def maxpool_backward(X, p, G):
    Y = maxpool_forward(X, p)
    up = lambda A: np.repeat(np.repeat(A, p, axis=0), p, axis=1)
    mask = (X == up(Y))
    return mask * up(G)

X_mp = rng.normal(size=(6, 6)); G_mp = rng.normal(size=(3, 3))
Y_mp = maxpool_forward(X_mp, 2)
dX_mp = maxpool_backward(X_mp, 2, G_mp)
```

</details>

### Exercise 13 · Testing equivariance numerically
*💻 Code · ★★☆*

Use circular padding (`np.pad(I, 1, mode='wrap')`) so borders do not interfere. For a random $12\times12$ image $I$
and $3\times3$ kernel:

1. `equiv_err`: max abs difference between `conv(roll(I))` and `roll(conv(I))` for a shift of (2, 3).
2. `gap_err`: the same for **global average pooling** of the conv output: `mean(conv(roll(I)))` vs `mean(conv(I))`.
3. `pool_err`: max abs difference between `maxpool(conv(roll(I)))` and `maxpool(conv(I))` for a shift of (1, 0) and $2\times2$ pooling.

Which of these is (exactly) zero?

In [ ]:
I_eq = rng.normal(size=(12, 12)); K_eq = rng.normal(size=(3, 3))
equiv_err = None
gap_err = None
pool_err = None

check('conv is shift-equivariant', None if equiv_err is None else equiv_err < 1e-10, True)
check('global average pooling is shift-invariant', None if gap_err is None else gap_err < 1e-10, True)
check('2x2 max-pool is NOT invariant to a 1-pixel shift', None if pool_err is None else pool_err > 1e-3, True)

<details>
<summary><b>💡 Hint</b></summary>

`circ = lambda I: conv2d(np.pad(I, 1, mode='wrap'), K_eq)` keeps the $12\times12$ size.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With circular padding the conv is exactly equivariant and global average pooling exactly invariant. Max-pooling with stride 2 is
only equivariant to shifts that are multiples of 2; a 1-pixel shift changes which values fall into each window, so it is only
*approximately* invariant. (This is the "CNNs are not truly shift-invariant" observation of Zhang, 2019.)

```python
I_eq = rng.normal(size=(12, 12)); K_eq = rng.normal(size=(3, 3))
circ = lambda I: conv2d(np.pad(I, 1, mode='wrap'), K_eq)
sh = lambda A, s: np.roll(A, s, axis=(0, 1))
equiv_err = np.abs(circ(sh(I_eq, (2, 3))) - sh(circ(I_eq), (2, 3))).max()
gap_err = abs(circ(sh(I_eq, (2, 3))).mean() - circ(I_eq).mean())
pool_err = np.abs(maxpool_forward(circ(sh(I_eq, (1, 0))), 2) - maxpool_forward(circ(I_eq), 2)).max()
print(equiv_err, gap_err, pool_err)
```

</details>

### Exercise 14 · Residual connections keep gradients alive
*💻 Code · ★★★*

Simulate a deep network of $L=50$ layers of width $d=32$ with weights $W_l \sim \mathcal N(0, 0.25/d)$:

- plain: $h_l = \tanh(W_l h_{l-1})$
- residual: $h_l = h_{l-1} + \tanh(W_l h_{l-1})$

Backpropagate by hand (Jacobian of $\tanh(Wh)$ is $\mathrm{diag}(1-\tanh^2)W$) to get $\lVert\partial (\mathbf 1^\top h_L)/\partial h_0\rVert$
for both. Store them in `g_plain` and `g_res`.

In [ ]:
L_depth, d = 50, 32
Ws = [rng.normal(0, np.sqrt(0.25 / d), size=(d, d)) for _ in range(L_depth)]
h0 = rng.normal(size=d)
g_plain = None
g_res = None

check('plain gradient vanishes (< 1e-6)', None if g_plain is None else g_plain < 1e-6, True)
check('residual gradient survives (> 1)', None if g_res is None else g_res > 1, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Forward pass: store every pre-activation $a_l = W_l h_{l-1}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Backward: start with $g = \mathbf 1$; plain: $g \leftarrow W_l^\top((1-\tanh^2 a_l)\odot g)$; residual: $g \leftarrow g + W_l^\top((1-\tanh^2 a_l)\odot g)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each plain layer multiplies the gradient by a matrix with spectral norm about $\sqrt{0.25}\cdot 2 \approx 1$ at best and
typically shrinks it (≈ 0.5 per layer here), so after 50 layers it is ~$10^{-15}$. The residual Jacobian $I + J_l$ keeps
the gradient of order 1 or larger. This is the core reason ResNets can be trained with 100+ layers.

```python
L_depth, d = 50, 32
Ws = [rng.normal(0, np.sqrt(0.25 / d), size=(d, d)) for _ in range(L_depth)]
h0 = rng.normal(size=d)

def grad_norm(residual):
    h, pre = h0.copy(), []
    for W in Ws:
        a = W @ h; pre.append(a)
        h = h + np.tanh(a) if residual else np.tanh(a)
    g = np.ones(d)
    for W, a in zip(reversed(Ws), reversed(pre)):
        back = W.T @ ((1 - np.tanh(a) ** 2) * g)
        g = g + back if residual else back
    return np.linalg.norm(g)

g_plain, g_res = grad_norm(False), grad_norm(True)
print(g_plain, g_res)
```

</details>

### Exercise 15 · 1-D convolution for time series
*💻 Code · ★☆☆*

The note mentions 1-D CNNs for audio/time series. A 1-D conv with kernel $(1/3, 1/3, 1/3)$ is a moving average.
Implement `conv1d(x, k)` (valid cross-correlation) and apply it to a noisy sine. Store the output in `smooth`.

In [ ]:
def conv1d(x, k):
    return None

t = np.linspace(0, 4 * np.pi, 200)
x_ts = np.sin(t) + 0.3 * rng.normal(size=200)
smooth = conv1d(x_ts, np.ones(3) / 3)

check('conv1d = np.correlate valid', smooth, np.correlate(x_ts, np.ones(3) / 3, mode='valid'))

<details>
<summary><b>💡 Hint</b></summary>

`np.array([x[i:i+len(k)] @ k for i in range(len(x)-len(k)+1)])`, or `np.lib.stride_tricks.sliding_window_view(x, len(k)) @ k`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The moving average is a low-pass filter; a learned 1-D CNN learns such filters (and many others) from data.

```python
def conv1d(x, k):
    return np.lib.stride_tricks.sliding_window_view(x, len(k)) @ k

t = np.linspace(0, 4 * np.pi, 200)
x_ts = np.sin(t) + 0.3 * rng.normal(size=200)
smooth = conv1d(x_ts, np.ones(3) / 3)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [CNN](CNN.md).*